In [1]:
%useLatestDescriptors
%use dataframe, kandy
%use kandy-geo
%use lets-plot
%use lets-plot-gt


In [2]:
USE {
    dependencies {
        implementation("org.xerial:sqlite-jdbc:3.49.1.0")
        implementation("ch.qos.logback:logback-classic:1.5.12")
    }
}

In [3]:
import java.sql.Connection
import java.sql.DriverManager

In [4]:
Class.forName("org.sqlite.JDBC")
val connection = DriverManager.getConnection("jdbc:sqlite:/Users/unchil/full-stack-task-manager/full-stack-task-manager.sqlite")

In [24]:
import kotlinx.datetime.*
import kotlinx.datetime.format.*

val now = Clock.System.now()

@OptIn(FormatStringsInDatetimeFormats::class)
val currentTime = now
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd HH:mm:ss")})

@OptIn(FormatStringsInDatetimeFormats::class)
val previous24Hour = now
    .minus(24, DateTimeUnit.HOUR)
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd HH:mm:ss")})

print("Current time : ${currentTime}, Previous time : ${previous24Hour}")

Current time : 2025-03-27 17:29:53, Previous time : 2025-03-26 17:29:53

In [23]:
import java.time.*
import java.time.format.DateTimeFormatter
import java.text.SimpleDateFormat

val formatter = DateTimeFormatter.ofPattern("YYYY-MM-dd HH:mm:ss")
val now = java.time.LocalDateTime.now()
val prevDay = now.minusHours(24)

print("Current time : ${now.format(formatter)}, Previous time : ${prevDay.format(formatter)}")

Current time : 2025-03-27 17:29:49, Previous time : 2025-03-26 17:29:49

In [25]:
val whereStmt_last24h  = "WHERE obs_datetime >= '${previous24Hour}' "
val sqlStmt = "SELECT * FROM Observation " + whereStmt_last24h
val df_list = DataFrame.readSqlQuery(connection, sqlStmt)
df_list.describe()

name,type,count,unique,nulls,top,freq,min,median,max
sta_cde,String,3330,44,0,bgj8a,141,bgj8a,fnm5b,wn087
sta_nam_kor,String,3330,44,0,기장,141,강릉,완도 가교,해남 화산
obs_dat,String,3330,2,0,2025-03-27,2409,2025-03-26,2025-03-27,2025-03-27
obs_tim,String,3330,47,0,18:00:00,71,00:00:00,11:30:00,23:30:00
repair_gbn,String,3330,1,0,1,3330,1,1,1
obs_lay,String,3330,3,0,1,2061,1,1,3
wtr_tmp,String,3330,93,0,10,195,10,11.8,9.9
dox,String?,3330,51,2625,9.9,51,10,13.2,9.9
sal,String?,3330,8,3236,34.9,33,32.1,32.4,34.9
obs_datetime,String,3330,47,0,2025-03-26 18:00:00,71,2025-03-26 17:30:00,2025-03-27 05:00:00,2025-03-27 16:30:00


In [26]:
val df_code = DataFrame.readSqlTable(connection, "Observatory")
df_code.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
sta_cde,String,84,84,0,fwdo5,1,null,null,bgj8a,fwdo5,ys002
sta_nam_kor,String,84,80,0,여수,2,null,null,강릉,영광,흑산도 대둔
bld_dat,String,84,64,0,2010-03-14,7,null,null,2003-11-25,2008-07-23,2024-05-13
end_dat,String?,84,38,44,2010-03-15,3,null,null,2005-01-20,2010-03-15,2024-05-25
gru_nam,String,84,3,0,남해,50,null,null,남해,남해,서해
lon,Double,84,84,0,126.736400,1,127.433104,1.137047,124.729500,127.236850,129.813100
lat,Double,84,84,0,34.382500,1,35.182755,1.251893,33.291000,34.741435,38.368100
sur_tmp_yn,String,84,2,0,Y,53,null,null,N,Y,Y
mid_tmp_yn,String,84,2,0,N,61,null,null,N,N,Y
bot_tmp_yn,String,84,2,0,N,75,null,null,N,N,Y


In [27]:
val df = df_list.innerJoinWith(df_code.select { sta_cde and gru_nam and lon and lat }) {
    right.sta_cde == sta_cde
}
df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
sta_cde,String,3330,44,0,bgj8a,141,null,null,bgj8a,fnm5b,wn087
sta_nam_kor,String,3330,44,0,기장,141,null,null,강릉,완도 가교,해남 화산
obs_dat,String,3330,2,0,2025-03-27,2409,null,null,2025-03-26,2025-03-27,2025-03-27
obs_tim,String,3330,47,0,18:00:00,71,null,null,00:00:00,11:30:00,23:30:00
repair_gbn,String,3330,1,0,1,3330,null,null,1,1,1
obs_lay,String,3330,3,0,1,2061,null,null,1,1,3
wtr_tmp,String,3330,93,0,10,195,null,null,10,11.8,9.9
dox,String?,3330,51,2625,9.9,51,null,null,10,13.2,9.9
sal,String?,3330,8,3236,34.9,33,null,null,32.1,32.4,34.9
obs_datetime,String,3330,47,0,2025-03-26 18:00:00,71,null,null,2025-03-26 17:30:00,2025-03-27 05:00:00,2025-03-27 16:30:00


In [28]:
df.filter{ gru_nam.equals("동해") and obs_lay.equals("1")  }
    .select{ sta_cde and sta_nam_kor and wtr_tmp }
    .convert { wtr_tmp }.with{ it.toFloat()}
    .groupBy{sta_cde and sta_nam_kor}
    .sortBy { sta_cde and  sta_nam_kor }
    .plot{
        layout {
            title = "관측지점별 일평균 표층 해수 정보"
            size = 1000 to 600
            //    theme = Theme.HIGH_CONTRAST_DARK
            x.axis {
                name = "관측지점"
            }
            y.axis {
                name = "온도"
                limits = 0.0..20.0
            }
        }

        boxplot("sta_nam_kor", "wtr_tmp") {
            boxes {
                borderLine.color = Color.BLUE
                fillColor("sta_nam_kor"){
                    scale = categoricalColorHue()
                    legend{
                        name= "관측 지점"
                    }
                }
            }
        }


    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ErIvQW"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일평균 표층 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[0.0,20.0]
},
"data":{
},
"ggsize":{
"width":1000.0,
"height":600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sta_nam_kor",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"wtr_tmp",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_hue",
"name":"관측 지점"
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"name":"관측지점",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"온도",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max",
"fill":"sta_nam_kor",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["bgj8a$기장","bgna3$강릉","bsc87$삼척","byd8a$영덕","byy87$양양","fggo3$고성 가진","fghe8$구룡포 하정"],
"min":[11.600000381469727,9.800000190734863,9.899999618530273,10.5,7.300000190734863,6.699999809265137,11.199999809265137],
"middle":[11.699999809265137,9.800000190734863,10.300000190734863,10.800000190734863,7.5,6.699999809265137,11.399999618530273],
"max":[11.899999618530273,10.0,10.399999618530273,11.100000381469727,7.900000095367432,6.699999809265137,11.600000381469727],
"lower":[11.699999809265137,9.800000190734863,10.0,10.699999809265137,7.5,6.699999809265137,11.300000190734863],
"upper":[11.899999618530273,9.899999618530273,10.399999618530273,11.0,7.699999809265137,6.699999809265137,11.5],
"x":["기장","강릉","삼척","영덕","양양","고성 가진","구룡포 하정"],
"sta_nam_kor":["기장","강릉","삼척","영덕","양양","고성 가진","구룡포 하정"]
},
"color":"#5470c6",
"sampling":"none",
"inherit_aes":false,
"position":{
"name":"dodge",
"width":1.0
},
"geom":"boxplot",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_cde"
},{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"lower"
},{
"type":"float",
"column":"middle"
},{
"type":"float",
"column":"upper"
},{
"type":"float",
"column":"max"
},{
"type":"str",
"column":"&merged_groups"
}]
}
},{
"mapping":{
"x":"x",
"y":"y",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["bgna3$강릉","bgna3$강릉","bgna3$강릉","bgna3$강릉","bgna3$강릉","bgna3$강릉","bgna3$강릉","bgna3$강릉","bgna3$강릉","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","byy87$양양","fggo3$고성 가진","fggo3$고성 가진","fggo3$고성 가진","fggo3$고성 가진"],
"x":["강릉","강릉","강릉","강릉","강릉","강릉","강릉","강릉","강릉","양양","양양","양양","양양","양양","양양","양양","양양","양양","양양","양양","양양","양양","양양","양양","고성 가진","고성 가진","고성 가진","고성 가진"],
"y":[9.600000381469727,10.399999618530273,10.300000190734863,10.5,10.399999618530273,10.399999618530273,10.199999809265137,10.199999809265137,10.100000381469727,8.0,8.0,8.100000381469727,8.100000381469727,8.0,8.0,8.0,7.199999809265137,7.099999904632568,7.099999904632568,7.099999904632568,7.099999904632568,7.099999904632568,7.0,7.0,6.599999904632568,6.599999904632568,6.599999904632568,6.599999904632568]
},
"sampling":"none",
"inherit_aes":false,
"position":{
"name":"dodge",
"width":1.0
},
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_cde"
},{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"&merged_groups"
}]
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_cde"
},{
"type":"str",
"column":"sta_nam_kor"
},{
"type

In [29]:
df.filter{ gru_nam.equals("서해") and obs_lay.equals("1")  }
    .select{ sta_cde and sta_nam_kor and wtr_tmp }
    .convert { wtr_tmp }.with{ it.toFloat()}
    .groupBy{sta_cde and sta_nam_kor}
    .sortBy { sta_cde and  sta_nam_kor }
    .plot{
        layout {
            title = "관측지점별 일평균 표층 해수 정보"
            size = 1000 to 600
            //    theme = Theme.HIGH_CONTRAST_DARK
            x.axis {
                name = "관측지점"
            }
            y.axis {
                name = "온도"
                limits = 0.0..20.0
            }
        }

        boxplot("sta_nam_kor", "wtr_tmp") {
            boxes {
                borderLine.color = Color.BLUE
                fillColor("sta_nam_kor"){
                    scale = categoricalColorHue()
                    legend{
                        name= "관측 지점"
                    }
                }
            }
        }


    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="UdFva9"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일평균 표층 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[0.0,20.0]
},
"data":{
},
"ggsize":{
"width":1000.0,
"height":600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sta_nam_kor",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"wtr_tmp",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_hue",
"name":"관측 지점"
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"name":"관측지점",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"온도",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max",
"fill":"sta_nam_kor",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["br001$태안 고남","egsi4$군산 신시도","emp67$목포","esafc$신안 압해","fbn69$백령도","fjdfc$진도 전두","fjh5a$해남 임하","fsch6$서산 창리","ftdk5$태안 대야도","ftpk5$태안 파도리","sj086$서산 지곡"],
"min":[7.400000095367432,7.699999809265137,9.0,9.600000381469727,6.400000095367432,8.899999618530273,9.0,9.399999618530273,7.699999809265137,5.699999809265137,6.800000190734863],
"middle":[7.699999809265137,8.0,9.5,10.100000381469727,6.800000190734863,9.100000381469727,9.199999809265137,9.600000381469727,7.699999809265137,8.0,8.699999809265137],
"max":[8.100000381469727,8.399999618530273,9.899999618530273,10.899999618530273,7.5,9.399999618530273,9.399999618530273,9.699999809265137,7.699999809265137,11.300000190734863,10.699999809265137],
"lower":[7.674999833106995,7.900000095367432,9.399999618530273,9.899999618530273,6.599999904632568,9.0,9.100000381469727,9.5,7.699999809265137,6.0,8.0],
"upper":[7.900000095367432,8.100000381469727,9.699999809265137,10.300000190734863,7.0,9.199999809265137,9.300000190734863,9.600000381469727,7.699999809265137,9.899999618530273,9.899999618530273],
"x":["태안 고남","군산 신시도","목포","신안 압해","백령도","진도 전두","해남 임하","서산 창리","태안 대야도","태안 파도리","서산 지곡"],
"sta_nam_kor":["태안 고남","군산 신시도","목포","신안 압해","백령도","진도 전두","해남 임하","서산 창리","태안 대야도","태안 파도리","서산 지곡"]
},
"color":"#5470c6",
"sampling":"none",
"inherit_aes":false,
"position":{
"name":"dodge",
"width":1.0
},
"geom":"boxplot",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_cde"
},{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"lower"
},{
"type":"float",
"column":"middle"
},{
"type":"float",
"column":"upper"
},{
"type":"float",
"column":"max"
},{
"type":"str",
"column":"&merged_groups"
}]
}
},{
"mapping":{
"x":"x",
"y":"y",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["esafc$신안 압해","esafc$신안 압해","fjh5a$해남 임하","fjh5a$해남 임하","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","fsch6$서산 창리","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도","ftdk5$태안 대야도"],
"x":["신안 압해","신안 압해","해남 임하","해남 임하","서산 창리","서산 창리","서산 창리","서산 창리","서산 창리","서산 창리","서산 창리","서산 창리","서산 창리","서산 창리","서산 창리","서산 창리","서산 창리","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 대야도","태안 

In [30]:
df.filter{ gru_nam.equals("남해") and obs_lay.equals("1")  }
    .select{ sta_cde and sta_nam_kor and wtr_tmp }
    .convert { wtr_tmp }.with{ it.toFloat()}
    .groupBy{sta_cde and sta_nam_kor}
    .sortBy { sta_cde and  sta_nam_kor }
    .plot{
        layout {
            title = "관측지점별 일평균 표층 해수 정보"
            size = 1000 to 600
            //    theme = Theme.HIGH_CONTRAST_DARK
            x.axis {
                name = "관측지점"
            }
            y.axis {
                name = "온도"
                limits = 0.0..20.0
            }
        }

        boxplot("sta_nam_kor", "wtr_tmp") {
            boxes {
                borderLine.color = Color.BLUE
                fillColor("sta_nam_kor"){
                    scale = categoricalColorHue()
                    legend{
                        name= "관측 지점"
                    }
                }
            }
        }


    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="wgpOn3"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일평균 표층 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[0.0,20.0]
},
"data":{
},
"ggsize":{
"width":1000.0,
"height":600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sta_nam_kor",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"wtr_tmp",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_hue",
"name":"관측 지점"
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"name":"관측지점",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"온도",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max",
"fill":"sta_nam_kor",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["ejhfc$장흥 회진","ejj47$서제주","eng5c$남해 강진","fgg4c$거제 가배","fgsj3$고흥 소록도","fhhfc$해남 화산","fnm5b$남해 미조","fth59$통영 학림","ftp4c$통영 풍화","ftsj3$통영 수월","fwbf1$완도 백도","fwdf1$완도 동백","fwdo5$완도 대창","fwgf1$완도 가교","fwih6$완도 일정","fwmg3$완도 망남","fwso5$완도 사동","fwyo5$완도 감목","gi086$거제 일운","km001$여수 신월","tb087$통영 비산도","ty004$통영 영운","ty005$통영 사량","wc001$완도 청산","wk094$완도 금일","wn087$완도 노화도"],
"middle":[11.399999618530273,14.800000190734863,11.5,13.0,10.0,10.199999809265137,9.800000190734863,10.699999809265137,10.699999809265137,10.800000190734863,10.0,10.800000190734863,11.100000381469727,10.899999618530273,10.100000381469727,9.399999618530273,10.899999618530273,10.300000190734863,12.199999809265137,11.100000381469727,11.399999618530273,11.199999809265137,10.100000381469727,10.199999809265137,10.0,9.5],
"min":[10.800000190734863,14.600000381469727,11.100000381469727,11.699999809265137,9.800000190734863,9.899999618530273,9.699999809265137,10.699999809265137,10.5,10.399999618530273,9.899999618530273,9.899999618530273,10.0,10.800000190734863,9.899999618530273,9.100000381469727,10.100000381469727,10.100000381469727,11.899999618530273,10.699999809265137,11.199999809265137,11.0,10.0,9.899999618530273,9.899999618530273,9.399999618530273],
"max":[12.399999618530273,15.399999618530273,12.300000190734863,13.5,10.100000381469727,10.399999618530273,10.199999809265137,10.699999809265137,11.0,11.5,10.199999809265137,11.899999618530273,12.300000190734863,11.5,10.399999618530273,9.699999809265137,12.399999618530273,10.399999618530273,12.5,11.600000381469727,11.800000190734863,11.5,10.300000190734863,11.0,10.100000381469727,9.699999809265137],
"upper":[11.800000190734863,15.0,11.800000190734863,13.199999809265137,10.0,10.199999809265137,10.0,10.699999809265137,10.800000190734863,11.050000190734863,10.100000381469727,11.0,11.5,11.100000381469727,10.199999809265137,9.5,11.650000095367432,10.300000190734863,12.399999618530273,11.300000190734863,11.600000381469727,11.399999618530273,10.199999809265137,10.5,10.0,9.600000381469727],
"lower":[11.199999809265137,14.699999809265137,11.199999809265137,12.5,9.899999618530273,10.0,9.800000190734863,10.699999809265137,10.600000381469727,10.699999809265137,10.0,10.399999618530273,10.300000190734863,10.800000190734863,10.0,9.300000190734863,10.5,10.199999809265137,12.100000381469727,10.899999618530273,11.399999618530273,11.100000381469727,10.100000381469727,10.100000381469727,9.899999618530273,9.5],
"x":["장흥 회진","서제주","남해 강진","거제 가배","고흥 소록도","해남 화산","남해 미조","통영 학림","통영 풍화","통영 수월","완도 백도","완도 동백","완도 대창","완도 가교","완도 일정","완도 망남","완도 사동","완도 감목","거제 일운","여수 신월","통영 비산도","통영 영운","통영 사량","완도 청산","완도 금일","완도 노화도"],
"sta_nam_kor":["장흥 회진","서제주","남해 강진","거제 가배","고흥 소록도","해남 화산","남해 미조","통영 학림","통영 풍화","통영 수월",

In [31]:
val df_East = df.filter {
    gru_nam.equals("동해") and obs_lay.equals("1")
}.add {
    "hour" from obs_tim.map {
        LocalTime.parse(it).hour
    }
}.select{
    sta_nam_kor and wtr_tmp and obs_tim  and gru_nam and sta_cde and obs_datetime and "hour"
}.convert { wtr_tmp }.with{ it.toFloat()}


df_East

sta_nam_kor,wtr_tmp,obs_tim,gru_nam,sta_cde,obs_datetime,hour
기장,11.700000,17:30:00,동해,bgj8a,2025-03-26 17:30:00,17
강릉,9.900000,17:30:00,동해,bgna3,2025-03-26 17:30:00,17
삼척,10.400000,17:30:00,동해,bsc87,2025-03-26 17:30:00,17
영덕,10.600000,17:30:00,동해,byd8a,2025-03-26 17:30:00,17
양양,8.000000,17:30:00,동해,byy87,2025-03-26 17:30:00,17
고성 가진,6.700000,17:30:00,동해,fggo3,2025-03-26 17:30:00,17
구룡포 하정,11.500000,17:30:00,동해,fghe8,2025-03-26 17:30:00,17
기장,11.700000,18:00:00,동해,bgj8a,2025-03-26 18:00:00,18
강릉,9.800000,18:00:00,동해,bgna3,2025-03-26 18:00:00,18
삼척,10.400000,18:00:00,동해,bsc87,2025-03-26 18:00:00,18


In [32]:
df_East
    .groupBy { sta_cde and sta_nam_kor and hour }
    .aggregate {
        min{wtr_tmp} into "min"
        max{wtr_tmp} into "max"
        mean{wtr_tmp} into "mean"
        min{obs_datetime} into "time"
    }
    .plot{
        layout {
            title = "관측지점별 일별 해수 정보"
            size = 2600 to 1200
        }

        ribbon {
            x("time") {
                axis.name = "측정시간"
            }

            yMin("min")
            yMax("max")

            alpha = 0.6
            borderLine.width = 0.0
            fillColor("sta_nam_kor"){
                legend {
                    name = "관측지점"
                }
            }

            tooltips(title = value(sta_nam_kor)){
                line("최저 ${value("min")}, 최고 ${value("max")}")
                line("측정시간", hour.tooltipValue("d"))
            }
        }

        line {
            x("time")
            y("mean")
            width = 1.0
            color = Color.BLUE
            tooltips(enable=false)
        }

        y.axis {
            limits = 4.0..12.0
            name = "수온 최저~최고"
        }

        facetWrap(nCol = 2) {
            facet(sta_nam_kor)
        }
    }


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Z9Sua7"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일별 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[4.0,12.0]
},
"data":{
"min":[11.699999809265137,9.899999618530273,10.399999618530273,10.600000381469727,8.0,6.699999809265137,11.5,11.699999809265137,9.600000381469727,10.199999809265137,10.5,8.0,6.699999809265137,11.399999618530273,11.800000190734863,9.800000190734863,10.0,10.5,8.0,6.699999809265137,11.399999618530273,11.800000190734863,10.300000190734863,9.899999618530273,10.5,7.800000190734863,6.699999809265137,11.399999618530273,11.899999618530273,10.399999618530273,9.899999618530273,10.699999809265137,7.900000095367432,6.699999809265137,11.399999618530273,11.899999618530273,10.199999809265137,9.899999618530273,10.800000190734863,7.699999809265137,6.699999809265137,11.5,11.800000190734863,10.0,9.899999618530273,10.699999809265137,7.599999904632568,6.699999809265137,11.399999618530273,11.699999809265137,9.899999618530273,10.0,10.699999809265137,7.599999904632568,6.699999809265137,11.399999618530273,11.800000190734863,9.899999618530273,10.199999809265137,10.699999809265137,7.599999904632568,6.699999809265137,11.399999618530273,11.699999809265137,9.800000190734863,10.100000381469727,10.699999809265137,7.599999904632568,6.699999809265137,11.300000190734863,11.600000381469727,9.800000190734863,10.100000381469727,10.600000381469727,7.5,6.699999809265137,11.199999809265137,11.600000381469727,9.800000190734863,10.0,10.699999809265137,7.5,6.699999809265137,11.199999809265137,11.600000381469727,9.800000190734863,10.100000381469727,10.699999809265137,7.5,6.699999809265137,11.199999809265137,11.699999809265137,9.800000190734863,10.300000190734863,10.899999618530273,7.5,6.699999809265137,11.199999809265137,11.699999809265137,9.800000190734863,10.300000190734863,10.800000190734863,7.5,6.599999904632568,11.300000190734863,11.699999809265137,9.800000190734863,10.300000190734863,10.899999618530273,7.5,6.599999904632568,11.300000190734863,11.899999618530273,9.800000190734863,10.300000190734863,10.800000190734863,7.5,6.699999809265137,11.300000190734863,11.800000190734863,9.800000190734863,10.399999618530273,10.600000381469727,7.5,6.699999809265137,11.300000190734863,11.899999618530273,9.800000190734863,10.399999618530273,11.0,7.400000095367432,6.699999809265137,11.300000190734863,11.699999809265137,9.800000190734863,10.399999618530273,11.0,7.300000190734863,6.699999809265137,11.399999618530273,11.699999809265137,9.800000190734863,10.399999618530273,11.0,7.099999904632568,6.699999809265137,11.5,11.699999809265137,9.800000190734863,10.399999618530273,10.899999618530273,7.099999904632568,6.699999809265137,11.5,11.699999809265137,9.800000190734863,10.300000190734863,11.0,7.099999904632568,6.699999809265137,11.600000381469727,11.699999809265137,9.800000190734863,10.399999618530273,11.100000381469727,7.0,6.599999904632568,11.5],
"hour":[17.0,17.0,17.0,17.0,17.0,17.0,17.0,18.0,18.0,18.0,18.0,18.0,18.0,18.0,19.0,19.0,19.0,19.0,19.0,19.0,19.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,22.0,22.0,22.0,22.0,22.0,22.0,22.0,23.0,23.0,23.0,23.0,23.0,23.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,13.0,13.0,13.0,13.0,13.0,13.0,13.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,15.0,15.0,1

In [33]:
df_East
    .select{  sta_nam_kor and wtr_tmp and obs_datetime   }
   // .convert{ obs_tim }.with{ LocalTime.parse(it) }
    .plot{

        layout {
            title = "동해 해수 정보"
            size = 2600 to 600
        }

        x(obs_datetime) { axis.name = "관측일시"}
        y(wtr_tmp) {axis.name ="표층수온"}
        y.axis.limits = 2.0..15.0
        line{
            color(sta_nam_kor){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측소명"
                }
            }
        }

        //  facetWrap(nRow = 3){ facet(gruNam)   }

    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ma5bUG"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"동해 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[2.0,15.0]
},
"data":{
"obs_datetime":["2025-03-26 17:30:00","2025-03-26 17:30:00","2025-03-26 17:30:00","2025-03-26 17:30:00","2025-03-26 17:30:00","2025-03-26 17:30:00","2025-03-26 17:30:00","2025-03-26 18:00:00","2025-03-26 18:00:00","2025-03-26 18:00:00","2025-03-26 18:00:00","2025-03-26 18:00:00","2025-03-26 18:00:00","2025-03-26 18:00:00","2025-03-26 18:30:00","2025-03-26 18:30:00","2025-03-26 18:30:00","2025-03-26 18:30:00","2025-03-26 18:30:00","2025-03-26 18:30:00","2025-03-26 18:30:00","2025-03-26 19:00:00","2025-03-26 19:00:00","2025-03-26 19:00:00","2025-03-26 19:00:00","2025-03-26 19:00:00","2025-03-26 19:00:00","2025-03-26 19:00:00","2025-03-26 19:30:00","2025-03-26 19:30:00","2025-03-26 19:30:00","2025-03-26 19:30:00","2025-03-26 19:30:00","2025-03-26 19:30:00","2025-03-26 19:30:00","2025-03-26 20:00:00","2025-03-26 20:00:00","2025-03-26 20:00:00","2025-03-26 20:00:00","2025-03-26 20:00:00","2025-03-26 20:00:00","2025-03-26 20:00:00","2025-03-26 20:30:00","2025-03-26 20:30:00","2025-03-26 20:30:00","2025-03-26 20:30:00","2025-03-26 20:30:00","2025-03-26 20:30:00","2025-03-26 20:30:00","2025-03-26 21:00:00","2025-03-26 21:00:00","2025-03-26 21:00:00","2025-03-26 21:00:00","2025-03-26 21:00:00","2025-03-26 21:00:00","2025-03-26 21:00:00","2025-03-26 21:30:00","2025-03-26 21:30:00","2025-03-26 21:30:00","2025-03-26 21:30:00","2025-03-26 21:30:00","2025-03-26 21:30:00","2025-03-26 21:30:00","2025-03-26 22:00:00","2025-03-26 22:00:00","2025-03-26 22:00:00","2025-03-26 22:00:00","2025-03-26 22:00:00","2025-03-26 22:00:00","2025-03-26 22:00:00","2025-03-26 22:30:00","2025-03-26 22:30:00","2025-03-26 22:30:00","2025-03-26 22:30:00","2025-03-26 22:30:00","2025-03-26 22:30:00","2025-03-26 22:30:00","2025-03-26 23:00:00","2025-03-26 23:00:00","2025-03-26 23:00:00","2025-03-26 23:00:00","2025-03-26 23:00:00","2025-03-26 23:00:00","2025-03-26 23:00:00","2025-03-26 23:30:00","2025-03-26 23:30:00","2025-03-26 23:30:00","2025-03-26 23:30:00","2025-03-26 23:30:00","2025-03-26 23:30:00","2025-03-26 23:30:00","2025-03-27 00:00:00","2025-03-27 00:00:00","2025-03-27 00:00:00","2025-03-27 00:00:00","2025-03-27 00:00:00","2025-03-27 00:00:00","2025-03-27 00:00:00","2025-03-27 00:30:00","2025-03-27 00:30:00","2025-03-27 00:30:00","2025-03-27 00:30:00","2025-03-27 00:30:00","2025-03-27 00:30:00","2025-03-27 00:30:00","2025-03-27 01:00:00","2025-03-27 01:00:00","2025-03-27 01:00:00","2025-03-27 01:00:00","2025-03-27 01:00:00","2025-03-27 01:00:00","2025-03-27 01:00:00","2025-03-27 01:30:00","2025-03-27 01:30:00","2025-03-27 01:30:00","2025-03-27 01:30:00","2025-03-27 01:30:00","2025-03-27 01:30:00","2025-03-27 01:30:00","2025-03-27 02:00:00","2025-03-27 02:00:00","2025-03-27 02:00:00","2025-03-27 02:00:00","2025-03-27 02:00:00","2025-03-27 02:00:00","2025-03-27 02:00:00","2025-03-27 02:30:00","2025-03-27 02:30:00","2025-03-27 02:30:00","2025-03-27 02:30:00","2025-03-27 02:30:00","2025-03-27 02:30:00","2025-03-27 02:30:00","2025-03-27 03:00:00","2025-03-27 03:00:00","2025-03-27 03:00:00","2025-03-27 03:00:00","2025-03-27 03:00:00","2025-03-27 03:00:00","2025-03-27 03:00:00","2025-03-27 03:30:00","2025-03-27 03:30:00","2025-03-27 03:30:00","2025-03-27 03:30:00","2025-03-27 03:30:00","2025-03-27 03:30:00","2025-03-27 03:30:00","2025-03-27 04:00:00","2025-03-27 04:00:00","2025-03-27 04:00:00","2025-03-27 04:00:00","2025-03-27 04:00:00","2025-03-27 04:00:00","2025-03-27 04:00:00","2025-03-27 04:30:00","2025-03-27 04:30:00","2025-03-27 04

In [34]:
val currentTime = df.obs_datetime.max()
currentTime

2025-03-27 16:30:00

In [35]:
import kotlinx.datetime.LocalTime

val df_Current = df.filter {
    obs_datetime.equals(currentTime)   and repair_gbn.equals("1")
  //  gru_nam.equals("동해")
}.select{
    sta_nam_kor and obs_lay and wtr_tmp and gru_nam and    sta_cde  and  lon and lat and obs_datetime
}.convert { wtr_tmp }.with{ it.toFloat()
}.sortBy{sta_nam_kor  and obs_lay }


In [36]:
df_Current.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
sta_nam_kor,String,70,43,0,강릉,3,null,null,강릉,완도 가교,해남 화산
obs_lay,String,70,3,0,1,43,null,null,1,1,3
wtr_tmp,Float,70,37,0,10.000000,5,9.921429,1.804816,5.700000,10.300000,14.600000
gru_nam,String,70,3,0,남해,37,null,null,남해,남해,서해
sta_cde,String,70,43,0,bgna3,3,null,null,bgj8a,fnm5b,wn087
lon,Double,70,43,0,128.949200,3,127.635790,1.180272,124.729500,127.415400,129.549700
lat,Double,70,43,0,37.799000,3,35.518570,1.388290,33.310400,34.806000,38.368100
obs_datetime,String,70,1,0,2025-03-27 16:30:00,70,null,null,2025-03-27 16:30:00,2025-03-27 16:30:00,2025-03-27 16:30:00


In [2]:
val df_current = DataFrame.readJson("http://127.0.0.1:7788/nifs/seawaterinfo/current")
df_current.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
sta_cde,String,68,43,0,bgj8a,3,null,null,bgj8a,fnm5b,wn087
sta_nam_kor,String,68,43,0,기장,3,null,null,강릉,완도 감목,해남 화산
obs_datetime,String,68,1,0,2025-03-28 17:00:00,68,null,null,2025-03-28 17:00:00,2025-03-28 17:00:00,2025-03-28 17:00:00
obs_lay,String,68,3,0,1,43,null,null,1,1,3
wtr_tmp,String,68,40,0,9.6,4,null,null,10,11.7,9.9
dox,String?,68,15,53,9.7,2,null,null,10.4,13.2,9.8
sal,String?,68,3,66,34.8,1,null,null,32.3,32.3,34.8
gru_nam,String,68,3,0,남해,37,null,null,남해,남해,서해
lon,Double,68,43,0,129.227000,3,127.571165,1.183756,124.729500,127.074550,129.549700
lat,Double,68,43,0,35.187000,3,35.418725,1.301085,33.310400,34.803000,38.368100


In [159]:
df_current.convert { wtr_tmp }.with{ it.toFloat()}
    .filter { gru_nam.equals("동해") }
    .plot{
        layout{
            title = "Sea Water Quality"
            size = 1000 to 400
        }
        bars{
            alpha = 0.5
            x("sta_nam_kor"){
                axis{
                    name = "관측지점"
                }
            }
            y("wtr_tmp"){
                scale = continuous(0.0..15.0)
                axis {
                    name = "온도 °C"
                }
            }
            fillColor("obs_lay"){
                scale = categorical(
                    listOf(Color.BLUE, Color.GREEN, Color.RED),
                    listOf( "3", "2", "1"),
                )
                legend{
                    name= "관측 수심"
                    breaksLabeled("1" to "표층", "2" to "중층", "3" to "저층")
                }
            }

        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="HcRG4J"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Sea Water Quality"
},
"mapping":{
},
"data":{
"wtr_tmp":[11.699999809265137,11.699999809265137,11.600000381469727,9.699999809265137,9.5,7.900000095367432,10.199999809265137,10.199999809265137,10.300000190734863,11.699999809265137,11.100000381469727,10.899999618530273,6.599999904632568,6.599999904632568,6.199999809265137,11.600000381469727],
"obs_lay":["1","2","3","1","2","3","1","2","3","1","2","3","1","2","3","1"],
"sta_nam_kor":["기장","기장","기장","강릉","강릉","강릉","삼척","삼척","삼척","영덕","영덕","영덕","고성 가진","고성 가진","고성 가진","구룡포 하정"]
},
"ggsize":{
"width":1000.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true,
"name":"관측지점"
},{
"aesthetic":"y",
"name":"온도 °C",
"limits":[0.0,15.0]
},{
"aesthetic":"fill",
"breaks":["1","2","3"],
"values":["#5470c6","#3ba272","#ee6666"],
"name":"관측 수심",
"limits":["3","2","1"],
"labels":["표층","중층","저층"]
}],
"layers":[{
"mapping":{
"x":"sta_nam_kor",
"y":"wtr_tmp",
"fill":"obs_lay"
},
"stat":"identity",
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"float",
"column":"wtr_tmp"
},{
"type":"str",
"column":"obs_lay"
}]
},
"spec_id":"45"
};
 var containerDiv = document.getElementById("HcRG4J");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1000.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 기장 
 
 
 
 
 
 
 
 
 강릉 
 
 
 
 
 
 
 
 
 삼척 
 
 
 
 
 
 
 
 
 영덕 
 
 
 
 
 
 
 
 
 고성 가진 
 
 
 
 
 
 
 
 
 구룡포 하정 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 14 
 
 
 
 
 
 
 
 
 Sea Water Quality 
 
 
 
 
 온도 °C 
 
 
 
 
 관측지점 
 
 
 
 
 
 
 
 
 관측 수심 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 저층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 중층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 표층

In [160]:
df_current.convert { wtr_tmp }.with{ it.toFloat()}
    .filter { gru_nam.equals("남해") }
    .plot{
        layout{
            title = "Sea Water Quality"
            size = 1000 to 400
        }
        bars{
            alpha = 0.5
            x("sta_nam_kor"){
                axis{
                    name = "관측지점"
                }
            }
            y("wtr_tmp"){
                scale = continuous(0.0..15.0)
                axis {
                    name = "온도 °C"
                }
            }
            fillColor("obs_lay"){
                scale = categorical(
                    listOf(Color.BLUE, Color.GREEN, Color.RED),
                    listOf( "3", "2", "1"),
                )
                legend{
                    name= "관측 수심"
                    breaksLabeled("1" to "표층", "2" to "중층", "3" to "저층")
                }
            }

        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="8oHbv4"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Sea Water Quality"
},
"mapping":{
},
"data":{
"wtr_tmp":[10.800000190734863,10.800000190734863,14.0,11.899999618530273,12.100000381469727,10.100000381469727,10.0,10.0,10.399999618530273,10.100000381469727,10.800000190734863,10.800000190734863,10.699999809265137,10.699999809265137,10.699999809265137,10.600000381469727,9.5,9.399999618530273,9.899999618530273,10.0,10.5,10.699999809265137,10.699999809265137,10.0,9.600000381469727,9.699999809265137,9.899999618530273,10.0,11.800000190734863,11.0,11.5,11.399999618530273,11.5,10.300000190734863,10.100000381469727,9.899999618530273,9.399999618530273],
"obs_lay":["1","2","1","1","1","1","1","2","1","2","1","2","3","1","2","1","1","2","1","2","1","1","2","1","1","2","1","1","1","1","1","2","1","1","1","1","1"],
"sta_nam_kor":["장흥 회진","장흥 회진","서제주","남해 강진","거제 가배","고흥 소록도","해남 화산","해남 화산","남해 미조","남해 미조","통영 학림","통영 학림","통영 학림","통영 풍화","통영 풍화","통영 수월","완도 백도","완도 백도","완도 동백","완도 동백","완도 대창","완도 가교","완도 가교","완도 일정","완도 망남","완도 망남","완도 사동","완도 감목","거제 일운","여수 신월","통영 비산도","통영 비산도","통영 영운","통영 사량","완도 청산","완도 금일","완도 노화도"]
},
"ggsize":{
"width":1000.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true,
"name":"관측지점"
},{
"aesthetic":"y",
"name":"온도 °C",
"limits":[0.0,15.0]
},{
"aesthetic":"fill",
"breaks":["1","2","3"],
"values":["#5470c6","#3ba272","#ee6666"],
"name":"관측 수심",
"limits":["3","2","1"],
"labels":["표층","중층","저층"]
}],
"layers":[{
"mapping":{
"x":"sta_nam_kor",
"y":"wtr_tmp",
"fill":"obs_lay"
},
"stat":"identity",
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"float",
"column":"wtr_tmp"
},{
"type":"str",
"column":"obs_lay"
}]
},
"spec_id":"48"
};
 var containerDiv = document.getElementById("8oHbv4");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1000.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 장흥 회진 
 
 
 
 
 
 
 
 
 서제주 
 
 
 
 
 
 
 
 
 남해 강진 
 
 
 
 
 
 
 
 
 거제 가배 
 
 
 
 
 
 
 
 
 고흥 소록도 
 
 
 
 
 
 
 
 
 해남 화산 
 
 
 
 
 
 
 
 
 남해 미조 
 
 
 
 
 
 
 
 
 통영 학림 
 
 
 
 
 
 
 
 
 통영 풍화 
 
 
 
 
 
 
 
 
 통영 수월 
 
 
 
 
 
 
 
 
 완도 백도 
 
 
 
 
 
 
 
 
 완도 동백 
 
 
 
 
 
 
 
 
 완도 대창 
 
 
 
 
 
 
 
 
 완도 가교 
 
 
 
 
 
 
 
 
 완도 일정 
 
 
 
 
 
 
 
 
 완도 망남 
 
 
 
 
 
 
 
 
 완도 사동 
 
 
 
 
 
 
 
 
 완도 감목 
 
 
 
 
 
 
 
 
 거제 일운 
 
 
 
 
 
 
 
 
 여수 신월 
 
 
 
 
 
 
 
 
 통영 비산도 
 
 
 
 
 
 
 
 
 통영 영운 
 
 
 
 
 
 
 
 
 통영 사량 
 
 
 
 
 
 
 
 
 완도 청산 
 
 
 
 
 
 
 
 
 완도 금일 
 
 
 
 
 
 
 
 
 완도 노화도 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 14 
 
 
 
 
 
 
 
 
 Sea Water Quality 
 
 
 
 
 온도 °C 
 
 
 
 
 관측지점 
 
 
 
 
 
 
 
 
 관측 수심 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 저층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 중층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 표층

In [161]:
df_current.convert { wtr_tmp }.with{ it.toFloat()}
    .filter { gru_nam.equals("서해") }
    .plot{
        layout{
            title = "Sea Water Quality"
            size = 1000 to 400
        }
        bars{
            alpha = 0.5
            x("sta_nam_kor"){
                axis{
                    name = "관측지점"
                }
            }
            y("wtr_tmp"){
                scale = continuous(0.0..15.0)
                axis {
                    name = "온도 °C"
                }
            }
            fillColor("obs_lay"){
                scale = categorical(
                    listOf(Color.BLUE, Color.GREEN, Color.RED),
                    listOf( "3", "2", "1"),
                )
                legend{
                    name= "관측 수심"
                    breaksLabeled("1" to "표층", "2" to "중층", "3" to "저층")
                }
            }

        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="E2H2yw"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Sea Water Quality"
},
"mapping":{
},
"data":{
"wtr_tmp":[7.599999904632568,8.0,9.100000381469727,9.199999809265137,5.699999809265137,8.800000190734863,8.699999809265137,9.100000381469727,9.100000381469727,9.0,7.699999809265137,7.699999809265137,8.300000190734863,8.100000381469727,9.0],
"obs_lay":["1","1","1","1","1","1","2","1","1","2","1","2","1","2","1"],
"sta_nam_kor":["태안 고남","군산 신시도","목포","신안 압해","백령도","진도 전두","진도 전두","해남 임하","서산 창리","서산 창리","태안 대야도","태안 대야도","태안 파도리","태안 파도리","서산 지곡"]
},
"ggsize":{
"width":1000.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true,
"name":"관측지점"
},{
"aesthetic":"y",
"name":"온도 °C",
"limits":[0.0,15.0]
},{
"aesthetic":"fill",
"breaks":["1","2","3"],
"values":["#5470c6","#3ba272","#ee6666"],
"name":"관측 수심",
"limits":["3","2","1"],
"labels":["표층","중층","저층"]
}],
"layers":[{
"mapping":{
"x":"sta_nam_kor",
"y":"wtr_tmp",
"fill":"obs_lay"
},
"stat":"identity",
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"float",
"column":"wtr_tmp"
},{
"type":"str",
"column":"obs_lay"
}]
},
"spec_id":"51"
};
 var containerDiv = document.getElementById("E2H2yw");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1000.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 태안 고남 
 
 
 
 
 
 
 
 
 군산 신시도 
 
 
 
 
 
 
 
 
 목포 
 
 
 
 
 
 
 
 
 신안 압해 
 
 
 
 
 
 
 
 
 백령도 
 
 
 
 
 
 
 
 
 진도 전두 
 
 
 
 
 
 
 
 
 해남 임하 
 
 
 
 
 
 
 
 
 서산 창리 
 
 
 
 
 
 
 
 
 태안 대야도 
 
 
 
 
 
 
 
 
 태안 파도리 
 
 
 
 
 
 
 
 
 서산 지곡 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 14 
 
 
 
 
 
 
 
 
 Sea Water Quality 
 
 
 
 
 온도 °C 
 
 
 
 
 관측지점 
 
 
 
 
 
 
 
 
 관측 수심 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 저층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 중층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 표층

In [6]:
fun<T> makePointGeoJSON( coordinates:List<Pair<Double, Double>>, properties:Map<String, List<T>>): String {
    val first_str = "{" + "\n" +
            "\t\"type\": \"FeatureCollection\"," + "\n" +
            "\t\"features\": [" + "\n"

    val end_str = "\t]" + "\n" +
            "}" + "\n"

    var features_str = ""

    coordinates.forEachIndexed { index,    pair ->

        val delimiter1 = if (index < coordinates.size - 1) "," else ""

        features_str += "\t\t{\n" +
                "\t\t\t\"type\": \"Feature\",\n" +
                "\t\t\t\"geometry\": {\n" +
                "\t\t\t\t\"type\": \"Point\",\n" +
                "\t\t\t\t\"coordinates\": [${pair.first}, ${pair.second}]\n" +
                "\t\t\t},\n" +
                "\t\t\t\"properties\": {\n"

        val propertieIterator = properties.entries.iterator()

        while (propertieIterator.hasNext()) {
            val (key, values) = propertieIterator.next()
            val delimiter2 = if (propertieIterator.hasNext()) "," else ""

            features_str += "\t\t\t\t\"${key}\": \"${values[index]}\"${delimiter2} \n"
        }

        features_str +=  "\t\t\t}\n" + "\t\t}${delimiter1}\n"

    }

    return first_str + features_str + end_str
}

In [165]:
val points = df_current.convert { wtr_tmp }.with{ it.toFloat()}.filter {
    obs_lay.equals("1")
}.select{
    sta_cde and sta_nam_kor and  lon and lat
}.sortBy{sta_cde and  sta_nam_kor
}.map { Pair(lon, lat) }.toList()
points.size

43

In [164]:
val properties = df_current.filter {
   obs_lay.equals("1")
}
    .add("tempBoundary"){
        when(wtr_tmp.toFloat()) {
            in 0.0..6.0 -> "Low"
            in 6.1..10.0 -> "Middle"
            in 10.1..16.0 -> "High"
            else -> "Unknown"
        }
    }
    .select {sta_cde and  sta_nam_kor and wtr_tmp and obs_datetime  and "tempBoundary"}
    .sortBy{sta_cde and  sta_nam_kor  }.toMap()

properties["sta_cde"]?.size

43

In [167]:
val pointGeoJSON = makePointGeoJSON(points, properties)

In [168]:
DataFrame.readJsonStr(pointGeoJSON)
    .writeJson("/Volumes/WorkSpace/Notebook/data/nifsPoint.json")

In [169]:
val southKorea = GeoDataFrame.readGeoJson("/Volumes/WorkSpace/Notebook/data/southkorea.geojson")
val staPoint = GeoDataFrame.readGeoJson("/Volumes/WorkSpace/Notebook/data/nifsPoint.json")

In [170]:
val southKoreaBounds: Envelope = southKorea.bounds().also {
    // Use JTS API for in-place envelope expansion
    it.expandBy(0.2)
}

In [171]:

southKorea.plot{
    layout{
        title = "Korea Sea Water Quality"
      //  size = 1400 to 1400
        //  theme = Theme.HIGH_CONTRAST_DARK
        style(Style.BW) {
            this.panel.background {
                fillColor =  Color.hex("#537ab5")
                borderLineColor = Color.hex("#EFC623")
                borderLineWidth = 6.0
            }
        }
    }

    geoMap{
        fillColor= Color.hex("#e5f5e0")
        borderLine {
            width = 0.5
            color = Color.hex("#a1d99b")
        }
    }

    withData(staPoint) {

        geoPoints {
            size = 5.0
            symbol = Symbol.CIRCLE_FILLED
            fillColor(tempBoundary)
            tooltips(title = value(sta_nam_kor)){
                line("수온 ${value(wtr_tmp)}°C")
                line("측정시간", value(obs_datetime))
            }
        }
    }


}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Jd1c4R"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Korea Sea Water Quality"
},
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"size":0.5,
"color":"#a1d99b",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"fill":"#e5f5e0",
"map":{
"geometry":["{\"type\":\"MultiPolygon\",\"coordinates\":[[[[128.3649194917,38.6243350685],[128.3947835223,38.5780740365],[128.4414167978,38.5058046474],[128.4506941248,38.4741070283],[128.4744572914,38.4260929476],[128.5627547951,38.2889672584],[128.6316023609,38.1449648288],[128.6428328922,38.1292992431],[128.6787215922,38.0943058017],[128.6924748566,38.0649275611],[128.8591414717,37.8767764335],[128.9238386781,37.8023134422],[129.0013126278,37.7343609539],[129.0122990573,37.7272403587],[129.0642195107,37.6788596209],[129.0669051816,37.6584333992],[129.0671492481,37.6338564489],[129.0733342121,37.6130232322],[129.1087345365,37.5973981199],[129.1189884678,37.5806338402],[129.1482039365,37.505112962],[129.1612247887,37.4876162483],[129.1768497994,37.4801699777],[129.1899519893,37.46938706],[129.2354435897,37.3982608671],[129.2627872718,37.370917],[129.2722274533,37.3558617958],[129.2758894805,37.3399111335],[129.2828068399,37.324448965],[129.3310653374,37.2821720069],[129.3464461534,37.2475039615],[129.3601180378,37.1717796949],[129.3789168242,37.1301129554],[129.4301863659,37.0730654479],[129.434092682,37.0611026724],[129.4282332864,37.0495466474],[129.4189559483,37.0385196388],[129.4130965445,37.0277367114],[129.412689633,37.018052448],[129.4199324954,36.9867617989],[129.4287215264,36.8974062755],[129.4379989231,36.8561466033],[129.4712019984,36.7718773096],[129.4738875737,36.7306175937],[129.463470895,36.6924502549],[129.4409285801,36.6577822907],[129.4226180537,36.6166446314],[129.4228621805,36.6149763138],[129.4279891043,36.5756696199],[129.4416609954,36.5346540236],[129.447113488,36.4933129127],[129.4409285498,36.4079449961],[129.4348250761,36.3895938351],[129.4039006176,36.3594425052],[129.3918563304,36.3424747029],[129.3865666203,36.322699319],[129.3823348485,36.201849707],[129.3857528032,36.1854515707],[129.3918563333,36.1763370125],[129.3985294907,36.1726748678],[129.4039005976,36.1676293183],[129.4096785794,36.1336123673],[129.4170841685,36.1075706822],[129.4251408289,36.1030134284],[129.4305119006,36.0973981528],[129.4267684105,36.0817731525],[129.4187118006,36.0729841455],[129.408295117,36.0689151043],[129.3984481031,36.066310922],[129.3817826673,36.0551746207],[129.3808587342,36.0414868754],[129.4019370429,36.0256600839],[129.4170806359,36.0119127132],[129.4330514795,35.9961077403],[129.4533314103,35.9932873685],[129.4790145222,36.0095889398],[129.5425724472,36.065904012],[129.5547949502,36.0824779008],[129.5712996819,36.0718448065],[129.574961804,36.0546736047],[129.5820418747,36.0368106353],[129.5872501971,36.016913155],[129.5716437692,35.9971243936],[129.5355895705,35.9390356641],[129.532839056,35.9130129127],[129.5221343282,35.8445522464],[129.4962165012,35.785073939],[129.497650567,35.747870147],[129.4711259106,35.6832223268],[129.4515757092,35.6518022926],[129.443053381,35.6360899906],[129.4513757628,35.6182433264],[129.4605899019,35.6092942117],[129.467275133,35.5996675541],[129.4634878981,35.5778833536],[129.461351289,35.5520

In [3]:
@file:DependsOn("org.geotools:gt-shapefile:[32.1]")
@file:DependsOn("org.geotools:gt-cql:[32.1]")

In [4]:
import org.geotools.api.data.FileDataStore
import org.geotools.api.data.FileDataStoreFinder
import org.geotools.api.feature.simple.SimpleFeatureType
import java.io.File
import org.geotools.feature.simple.SimpleFeatureTypeBuilder
import org.geotools.referencing.crs.DefaultGeographicCRS
import org.geotools.feature.simple.SimpleFeatureBuilder
import org.geotools.feature.simple.SimpleFeatureImpl
import org.geotools.data.collection.ListFeatureCollection
import org.geotools.geometry.jts.JTSFactoryFinder
import org.locationtech.jts.geom.Point
import org.geotools.data.shapefile.ShapefileDataStoreFactory
import org.geotools.data.simple.SimpleFeatureCollection

In [5]:
var dataStore: FileDataStore = FileDataStoreFinder.getDataStore(File("/Users/unchil/Downloads/south_korea_Boundary.shp"))
val koreaFeatures:SimpleFeatureCollection = dataStore.featureSource.features
val korea = koreaFeatures.toSpatialDataset(10)
dataStore.schema

SimpleFeatureTypeImpl south_korea_Boundary identified extends polygonFeature(the_geom:MultiPolygon)

In [7]:
data class SeawaterInformationByObservationPoint(
    val sta_cde: String,
    val sta_nam_kor: String,
    val obs_datetime: String,
    val obs_lay: String,
    val wtr_tmp: String,
    val dox: String?,
    val sal: String?,
    val gru_nam: String,
    val lon: Double,
    val lat: Double,
)

In [9]:
val dfCurrentList = df_current.filter {
    obs_lay.equals("1")
}.toListOf<SeawaterInformationByObservationPoint>()


In [10]:

fun createFeatureType(): SimpleFeatureType {
    val builder = SimpleFeatureTypeBuilder()
    builder.setName( "Location" )

    builder.add("the_geom", org.locationtech.jts.geom.Point::class.java, DefaultGeographicCRS.WGS84) // 경도, 위도
    builder.add("wtr_tmp", String::class.java)
    builder.add("obs_datetime", String::class.java)
    builder.add("sta_nam_kor", String::class.java)
    builder.add("sta_cde", String::class.java)
    builder.add("tempBoundary",String::class.java)

    return builder.buildFeatureType()
}


fun createFeatureCollection(seaWaterInfo: List<SeawaterInformationByObservationPoint>): SimpleFeatureCollection {
    val featureType = createFeatureType()
    val featureBuilder = SimpleFeatureBuilder(featureType)
    val featureCollection = ListFeatureCollection(featureType)
    val geometryFactory = JTSFactoryFinder.getGeometryFactory()

    seaWaterInfo.forEach { info ->

        val point: Point = geometryFactory.createPoint(org.locationtech.jts.geom.Coordinate(info.lon, info.lat))
        featureBuilder.add(point)
        featureBuilder.add(info.wtr_tmp)
        featureBuilder.add(info.obs_datetime)
        featureBuilder.add(info.sta_nam_kor)
        featureBuilder.add(info.sta_cde)
        featureBuilder.add(
            when(info.wtr_tmp.toFloat()) {
                in 0.0..6.0 -> "Low"
                in 6.1..10.0 -> "Middle"
                in 10.1..16.0 -> "High"
                else -> "Unknown"
            })

        val feature = featureBuilder.buildFeature(null)
        featureCollection.add(feature)
    }
    return featureCollection
}

In [11]:
val pointData = createFeatureCollection(dfCurrentList).toSpatialDataset(10)

In [12]:
val southKoreaBounds = koreaFeatures.bounds
southKoreaBounds.expandBy(1.0)
val southKoreaLimits = coordMap(
    xlim = southKoreaBounds.minX to southKoreaBounds.maxX,
    ylim = southKoreaBounds.minY to southKoreaBounds.maxY
)

In [14]:
letsPlot() +
        geomPolygon(map = korea, fill = "white", color = "gray") +
        geomPoint(
            map=pointData ,
            size = 3,
            shape = 1,
            tooltips = layerTooltips()
                .line("수집시간|@obs_datetime")
                .line("관측지점|@sta_nam_kor/@sta_cde")
                .line("온도|@wtr_tmp °C" )
        ){ color="tempBoundary" } +
        labs( title="Korea EastSea 수온 정보", color="수온범위", caption="Nifs") +
        southKoreaLimits +
        ggsize(800, 800)

<path d="M244.80021146648687 576.6713893187316 L244.80021146648687 576.6713893187316 L246.8273415405656 578.9097619990443 L250.50790407784189 580.3727842085841 L251.62510693686272 581.5080221732601 L252.37611543301682 584.2468115955066 L253.11471065736987 585.3182540543617 L254.15743328308054 584.1203122546522 L254.73465470037263 585.3070939340078 L253.11471065736987 586.5606483607003 L253.6360719702261 588.4313118616692 L252.59334933688842 590.0487268712532 L248.62607975212632 593.7124997243982 L246.67556878595315 597.4542133372065 L246.11357301997123 599.4492370881189 L243.50676644044324 600.0507884976728 L241.50351172956107 599.8742405592384 L238.57876394237246 602.1865015356548 L230.87044453855742 603.604292923605 L228.29225517570376 605.798195734335 L225.13542994982527 606.5005538891119 L221.7553521154423 606.1622721327544 L217.89264489274865 606.8829493860317 L214.68143519012847 605.9168450836055 L207.43367919425145 606.0208211012477 L205.17332672007797 607.0793873722382 L203.82744653611917 609.4654591204444 L202.59852340721045 609.7213591653249 L201.36960034694494 609.1316640752357 L199.633268526346 606.3168089746041 L197.32032485051786 605.5299548407129 L195.07602451796993 603.176638714006 L194.2939824762325 600.3107040860568 L196.9380291686066 594.2933381302314 L200.38273788668266 591.9111343538166 L202.64661261734727 587.9725362450527 L204.72784959183082 587.4386595860678 L205.69020161527988 585.3579993121411 L217.03981155727888 581.8919305666932 L218.98879366851907 580.2109346438647 L226.4248441205109 579.4309143882992 L231.486255152995 577.2217511899098 L232.69844538993675 577.56497583605 L242.4354516367457 576.2726041261226 L244.80021146648687 576.6713893187316 ZM280.82134044150916 530.5892850241084 L280.82134044150916 530.5892850241084 L280.8772006145864 532.7878882058062 L281.8454430146103 534.3980565858205 L282.8385121183983 534.9484332953789 L282.9440257447586 535.7159007142095 L280.82134044150916 534.9484332953789 L279.68551759852016 532.7691633306567 L279.6296574254411 530.9601320056081 L280.82134044150916 530.5892850241084 ZM116.213919257405 531.5219879293586 L116.213919257405 531.5219879293586 L115.64290447601343 531.8703181924034 L112.7630038958032 529.6414846523662 L112.57680340023398 528.9258754101911 L113.5326325208298 527.4157616567122 L114.37674130290361 527.700569795893 L115.04706293747404 528.5324495930804 L116.213919257405 531.5219879293586 ZM225.31498065595406 520.7160215098802 L225.31498065595406 520.7160215098802 L227.68593327137933 520.7160215098802 L228.98312991380408 521.7924176125716 L228.79692948687443 522.2011842259503 L228.20729465287513 521.8674223157359 L228.07074754957102 522.2274344576681 L226.3763232732581 521.7511647859283 L225.7060016386913 522.6474276747535 L222.95023467643114 524.6045970688888 L221.49166436422456 524.5408656901991 L220.67858899045314 523.461100074318 L220.62272881737772 522.0661810789347 L221.2930505205859 520.6972676917494 L222.64610726744831 519.7520142727067 L223.80675688283736 519.7520142727067 L225.31498065595406 520.7160215098802 ZM232.75058603245634 525.2831176307404 L232.75058603245634 525.2831176307404 L231.99957746766086 525.4892862838947 L230.7147942343181 524.4508908929697 L230.54721379135663 523.2136326485565 L231.19891543433187 519.9020621323225 L230.54721379135663 519.0092345085245 L231.21753556320618 518.0862881635626 L233.16022707542652 518.7578737766476 L233.34022086645564 519.6582328422032 L232.61403906643682 520.2846711125426 L232.48369866767462 520.9860721519817 L233.8243420664694 522.7524224570011 L233.24712064917912 523.0823967427064 L232.75058603245634 525.2831176307404 ZM252.1650882413105 520.8698015375599 L252.1650882413105 520.8698015375599 L251.68096704129857 521.7286632106116 L250.78720479831463 522.2124343018659 L249.65138194770043 522.3099350098505 L248.1245380457076 521.2523629316706 L247.3611162014895 521.7736663883552 L247.07560877647302 521.6049028949706 L246.76527466294647 520.1871452008777 L247.6590369745736 517.65479189395 L